In [1]:

import pandas as pd
from ontosearch.onto import ontolister, ontocontext
from ontosearch.csv_importer import load_data
from ontosearch.find import matcher
from ontosearch.onto_api import bioportal_search, unpack_superclass
from ontosearch.rdf_print import term_lookup
from json import dump

/Users/pranavsingh/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
material_csv = load_data("csvs/material/material.csv")
# chear.owl file has some dependency issues regarding ontologies that it cannot access
# bero.owl is creating more files? i dont know why + it also has repeat ontologies

#look into cpdat.owl


In [3]:
onto = ontolister(
 onto_dir = ('./owlfiledemo/'))

Error loading ontology chear.owl: Cannot download 'http://hadatac.org/ont/hasco/'!

chear.owl load time: 6.4978938329732046

Error loading ontology bero.owl: 'http://purl.obolibrary.org/obo/NCIT_C103138' belongs to more than one entity types (cannot be both a property and a class/an individual)!

bero.owl load time: 83.0363311250112

Ontologies in Dictionary (keys): dict_keys([])TOTAL TIME: 89.53474954201374


In [4]:
#with open('full_onto.json', 'w') as file:
     #dump(onto, file)

In [5]:
df = pd.read_csv('../ontosearcherCPSC/csvs/material/material.csv')
#df1 = pd.read_csv('ontosearcher_v2/FullCSVs/material/material.csv')
df.dropna(axis=0, how='all', inplace=True)
df.reset_index(drop=True, inplace=True)

In [6]:
df

,Product Category,Product Subcategory,Nanomaterial Type,Reason Nano,Brand/Manufacturer,Manufacturers,Websites,Children's Use (1-3),Exposure (1-10),Toxicity (1-10),Public Perception (1-5),Stakeholder Perception (1-5),Relative Level of Concern (RLC),Prioritization Score Using Tool
0,Toys / Products for Children,Baby bottle/cup,Silver,antibacterial,BabyDream,BabyDream,http://babydream.koreasme.com/product02a.html,NaN,NaN,NaN,NaN,NaN,NaN,58.66
1,Toys / Products for Children,Baby bottle/cup,Silver,antibacterial,BabyDream,BabyDream,http://babydream.koreasme.com/product02a.html,NaN,NaN,NaN,NaN,NaN,NaN,58.66
2,Toys / Products for Children,Toy,Titanium dioxide (2-3nm),Unknown,NanoBioNet,NanoBioNet e.V.,http://www.nms-nano.com/en/translate-to-englis...,NaN,NaN,NaN,NaN,NaN,NaN,56.89
3,Tools - Personal Care,Brushes,Silver,antibacterial,Mouthwatchers,Mouthwatchers,http://www.mouthwatchers.com/collections/super...,NaN,NaN,NaN,NaN,NaN,NaN,55.22
4,Household - hardware tools,Towel,Nanosilver,dries quickly,Nano Cyclic,Nano Cyclic,https://www.amazon.com/nano-cyclic-microfiber-...,NaN,NaN,NaN,NaN,NaN,NaN,54.50
5,Electronics,Battery,Nano titanium dioxide,improved product,Toshiba,Toshiba,https://www.scib.jp/en/,NaN,NaN,NaN,NaN,NaN,NaN,53.65
6,Air Purifier /Filtration including face masks,Filter,Silver,antibacterial,Alkaline,Alkaline,http://www.cutcat.com/item/alkastick/1085/pgc1,NaN,NaN,NaN,NaN,NaN,NaN,53.18
7,Household - hardware tools,Kitchenware,Silver,antibacterial,Amore TM,Amore TM Kitchenware,http://www.amazon.com/namekiss-ceramic-innovat...,NaN,NaN,NaN,NaN,NaN,NaN,53.08
8,Household - hardware tools,Kitchenware,Silver,antibacterial,Amore TM,Amore TM Kitchenware,https://amorekitchenware.com/index.php/flameki...,NaN,NaN,NaN,NaN,NaN,NaN,53.08
9,Sports Equipment,Racquet,Fullerene - nano carbon,stronger,Yonex,Yonex,https://www.badmintonalley.com/Yonex_Nano_Spee...,NaN,NaN,NaN,NaN,NaN,NaN,52.92


In [7]:
isA_dict = {"material":["MaterialID", # identifying column
                         "http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C62371", #IRI
                         "nanomaterial"], #label
}

In [8]:
import ontosearch.curator as cur
from rdflib import Literal, URIRef, Graph
import re

def sanitize_uri(text):
    """
    Sanitize text to be used in URI by:
    1. Converting to lowercase
    2. Replacing spaces and special chars with underscores
    3. Removing parentheses and other invalid characters
    4. Removing multiple consecutive underscores
    """
    # Convert to lowercase
    text = text.lower()
    
    # Replace spaces and special characters with underscores
    text = re.sub(r'[^a-z0-9]+', '_', text)
    
    # Remove leading/trailing underscores
    text = text.strip('_')
    
    # Replace multiple consecutive underscores with a single one
    text = re.sub(r'_+', '_', text)
    
    return text

def csv_to_rdf_graph(dataframe, table_name, isA_type_uri, isA_type_label, id_column):
    """
    Convert a single CSV file to RDF graph with sanitized URIs
    
    Parameters:
    - dataframe: pandas DataFrame containing the CSV data
    - table_name: string name for the table (used in URI generation)
    - isA_type_uri: URI string for the rdf:type relation
    - isA_type_label: Human readable label for the type
    - id_column: Name of the column used for identification
    """
    # Initialize Graph object
    graph = Graph()
    
    # Sanitize column names in the dataframe
    dataframe.columns = [sanitize_uri(col) for col in dataframe.columns]
    
    # Sanitize table name
    table_name = sanitize_uri(table_name)
    
    # Sanitize id_column name
    id_column = sanitize_uri(id_column)
    
    # Convert dataframe to RDF with temporary IRIs
    cur.df_to_rdf(dataframe, table_name, graph)
    
    # Generate required IRIs and literals
    isA_ID_col = cur.make_example_IRI([table_name, "col", id_column])
    isA_IRI = URIRef(isA_type_uri)
    isA_label = Literal(isA_type_label)
    
    # Add type relationships for each row
    for s, p, o in graph.triples((None, isA_ID_col, None)):
        # Verify IRI is for a row
        if f"example.org/row/{table_name.lower()}" in s:
            # Add rdf:type triple
            graph.add((s,
                         URIRef("http://www.w3.org/1999/02/22-rdf-syntax-ns#type"),
                         isA_IRI))
            
            # Add label for the type (only needs to be done once)
            graph.add((isA_IRI,
                         URIRef("http://www.w3.org/2000/01/rdf-schema#label"),
                         isA_label))
    
    return graph

In [9]:
import pandas as pd

# Read your CSV file
df = pd.read_csv('../ontosearcherCPSC/csvs/material/material.csv')

# Original column names will be automatically sanitized, for example:
# 'Product Category' -> 'product_category'
# 'Children's Use (1-3)' -> 'childrens_use_1_3'
# 'Relative Level of Concern (RLC)' -> 'relative_level_of_concern_rlc'

# Define parameters
table_name = 'Material'  # Will be sanitized to 'material'
type_uri = 'http://example.org/ontology/Material'
type_label = 'Material'
id_column = 'material_id'  # Use the sanitized version of your ID column name

# Create the graph
graph = csv_to_rdf_graph(df, table_name, type_uri, type_label, id_column)

# Check the size of the graph
print(len(graph))

196


In [10]:
graph.serialize(format='turtle', destination='cpscdemo.ttl')

<Graph identifier=Nf437c3f1fc50406ea61feba59465ae02 (<class 'rdflib.graph.Graph'>)>

In [11]:
# import json 
# with open('full_onto.json', 'r') as f:
#     data = json.load(f)

# # Pretty print the JSON data
# formatted_json = json.dumps(data, indent=4)

# # Print the formatted JSON
# print(formatted_json)

In [12]:
newPrefixes = [
    # These are from common ontologies, mostly the ones we have downloaded 
    # for other exercises with the Ontosearch `matcher` function
    ("ncit","http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#"),
    ("obo","http://purl.obolibrary.org/obo/"),
    ("dcterms","http://purl.org/dc/terms/"),
    ("npo","http://purl.bioontology.org/ontology/npo#"),
    ("dc","http://purl.org/dc/elements/1.1/"),
    ("enm","http://purl.enanomapper.org/onto/"),
    ("sio","http://semanticscience.org/resource/"),
    ("edam","http://edamontology.org/"),
    ("birnlex","http://bioontology.org/projects/ontologies/birnlex#"),
    ("oboInOwl","http://www.geneontology.org/formats/oboInOwl#"),
    ("bao", "http://www.bioassayontology.org/bao#"),
    ("blkv", "https://w3id.org/biolink/vocab/"),
    ("caro", "http://purl.obolibrary.org/obo/caro#"),
    ("dcn", "http://dicom.nema.org/resources/ontology/DCM/"),
    ("sno", "http://snomed.info/id/")
]

for newOne, oldOne in newPrefixes:
    graph.bind(newOne, oldOne, override=True, replace=True)

[ns for ns in graph.namespaces()]

[('brick', rdflib.term.URIRef('https://brickschema.org/schema/Brick#')),
 ('csvw', rdflib.term.URIRef('http://www.w3.org/ns/csvw#')),
 ('dc', rdflib.term.URIRef('http://purl.org/dc/elements/1.1/')),
 ('dcat', rdflib.term.URIRef('http://www.w3.org/ns/dcat#')),
 ('dcmitype', rdflib.term.URIRef('http://purl.org/dc/dcmitype/')),
 ('dcterms', rdflib.term.URIRef('http://purl.org/dc/terms/')),
 ('dcam', rdflib.term.URIRef('http://purl.org/dc/dcam/')),
 ('doap', rdflib.term.URIRef('http://usefulinc.com/ns/doap#')),
 ('foaf', rdflib.term.URIRef('http://xmlns.com/foaf/0.1/')),
 ('geo', rdflib.term.URIRef('http://www.opengis.net/ont/geosparql#')),
 ('odrl', rdflib.term.URIRef('http://www.w3.org/ns/odrl/2/')),
 ('org', rdflib.term.URIRef('http://www.w3.org/ns/org#')),
 ('prof', rdflib.term.URIRef('http://www.w3.org/ns/dx/prof/')),
 ('prov', rdflib.term.URIRef('http://www.w3.org/ns/prov#')),
 ('qb', rdflib.term.URIRef('http://purl.org/linked-data/cube#')),
 ('schema', rdflib.term.URIRef('https://sc

In [13]:
graph = graph.serialize(format="turtle", destination="cpscgraph2demo.ttl")

In [14]:
df

,product_category,product_subcategory,nanomaterial_type,reason_nano,brand_manufacturer,manufacturers,websites,children_s_use_1_3,exposure_1_10,toxicity_1_10,public_perception_1_5,stakeholder_perception_1_5,relative_level_of_concern_rlc,prioritization_score_using_tool
0,Toys / Products for Children,Baby bottle/cup,Silver,antibacterial,BabyDream,BabyDream,http://babydream.koreasme.com/product02a.html,NaN,NaN,NaN,NaN,NaN,NaN,58.66
1,Toys / Products for Children,Baby bottle/cup,Silver,antibacterial,BabyDream,BabyDream,http://babydream.koreasme.com/product02a.html,NaN,NaN,NaN,NaN,NaN,NaN,58.66
2,Toys / Products for Children,Toy,Titanium dioxide (2-3nm),Unknown,NanoBioNet,NanoBioNet e.V.,http://www.nms-nano.com/en/translate-to-englis...,NaN,NaN,NaN,NaN,NaN,NaN,56.89
3,Tools - Personal Care,Brushes,Silver,antibacterial,Mouthwatchers,Mouthwatchers,http://www.mouthwatchers.com/collections/super...,NaN,NaN,NaN,NaN,NaN,NaN,55.22
4,Household - hardware tools,Towel,Nanosilver,dries quickly,Nano Cyclic,Nano Cyclic,https://www.amazon.com/nano-cyclic-microfiber-...,NaN,NaN,NaN,NaN,NaN,NaN,54.50
5,Electronics,Battery,Nano titanium dioxide,improved product,Toshiba,Toshiba,https://www.scib.jp/en/,NaN,NaN,NaN,NaN,NaN,NaN,53.65
6,Air Purifier /Filtration including face masks,Filter,Silver,antibacterial,Alkaline,Alkaline,http://www.cutcat.com/item/alkastick/1085/pgc1,NaN,NaN,NaN,NaN,NaN,NaN,53.18
7,Household - hardware tools,Kitchenware,Silver,antibacterial,Amore TM,Amore TM Kitchenware,http://www.amazon.com/namekiss-ceramic-innovat...,NaN,NaN,NaN,NaN,NaN,NaN,53.08
8,Household - hardware tools,Kitchenware,Silver,antibacterial,Amore TM,Amore TM Kitchenware,https://amorekitchenware.com/index.php/flameki...,NaN,NaN,NaN,NaN,NaN,NaN,53.08
9,Sports Equipment,Racquet,Fullerene - nano carbon,stronger,Yonex,Yonex,https://www.badmintonalley.com/Yonex_Nano_Spee...,NaN,NaN,NaN,NaN,NaN,NaN,52.92
